# Baseline - Bank Churn Prediction

Goal: establish a baseline, the reference point (floor) we compare against when we apply imbalance techniques and tuning in 02. 

Main metric: PR-AUC, it cares about the churners (the small group we actually want to catch - 20% of all data) and doesn't get flattered by the huge crowd that obviously stays. Recall, precision and ROC-AUC are there for context.

What we do:
- creating 2 baseline models: Logistic Regression and Random Forest
- check PR-AUC, Recall, Precision, ROC-AUC
- build confusion matrix at the default 0.5 threshold, to show it misses many churners
- test RF model without dropping Complain column, to visualize that it will achieve ~100% PR-AUC (impossibly good accuracy, model won't provide any useful business insights)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv("../../data/Customer-Churn-Records.csv")

In [ ]:
df.shape

In [ ]:
df.columns

## Feature Selection

Several columns were removed before training.

### Removed features

- **RowNumber** - unique row identifier. It does not contain information that can generalize to unseen customers.
- **CustomerId** - unique customer identifier. The model cannot learn meaningful churn patterns from IDs.
- **Surname** - customer surname. It is highly specific to individual customers and is unlikely to provide an useful signal.
- **Complain** - removed because it introduces data leakage. A complaint is often recorded shortly before churn occurs, making it unavailable in a real-world prediction scenario.

### Target variable

The target variable is:

- **Exited** = 1 if the customer left the bank.
- **Exited** = 0 otherwise.

In [ ]:
X = df.drop(columns=["Complain", "Exited", "RowNumber", "CustomerId", "Surname"])
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
X.head()

In [ ]:
categorical = X.select_dtypes(include='object').columns
numerical = X.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown="ignore"), categorical),
        ('scaler', StandardScaler(), numerical)
    ],
    remainder='passthrough'
)

lr_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

scores = cross_validate(lr_pipeline, X_train, y_train, cv=5, scoring=['average_precision', 'roc_auc', 'f1', 'recall', 'precision'])

print("PR-AUC mean: ", scores['test_average_precision'].mean())
print("ROC-AUC mean: ", scores['test_roc_auc'].mean())
print("Recall mean: ", scores['test_recall'].mean())
print("F1 mean: ", scores['test_f1'].mean())
print("Precision mean: ", scores['test_precision'].mean())

In [ ]:
rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_scores = cross_validate(rf_pipeline, X_train, y_train, cv=5, scoring=['average_precision', 'roc_auc', 'f1', 'recall', 'precision'])

print("PR-AUC mean: ", rf_scores['test_average_precision'].mean())
print("ROC-AUC mean: ", rf_scores['test_roc_auc'].mean())
print("Recall mean: ", rf_scores['test_recall'].mean())
print("F1 mean: ", rf_scores['test_f1'].mean())
print("Precision mean: ", rf_scores['test_precision'].mean())

## Baseline Comparison: Logistic Regression vs Random Forest

PR-AUC was selected as the main metric because the dataset is imbalanced and accuracy can be misleading.

| Model | PR-AUC |
|---------|---------|
| Logistic Regression | ~0.46 |
| Random Forest | ~0.67 |

For PR-AUC, a random (no-skill) model scores the positive-class rate. Churn here is ~20% (see EDA), so ~0.20 is the floor - both models clear it.

Both models perform better than random guessing (~0.20 PR-AUC), but Random Forest achieves better results. It can capture more complex relationships between features, such as non-linear effects and feature interactions (e.g. churn spikes for customers with 3-4 products). So it will be used for further optimization.

In [ ]:
y_pred = cross_val_predict(rf_pipeline, X_train , y_train, cv=5)
cm = confusion_matrix(y_train, y_pred)
cm

## The Default Threshold Misses Many Churners

The confusion matrix reveals that the model misses a large number of churners when using the default classification threshold of 0.5.

### Results

- Total churners: 1,630
- Correctly identified churners: 697
- Missed churners: 933

This means that approximately 57% of churning customers are not detected.

In [ ]:
X_leak = df.drop(columns=["Exited", "RowNumber", "CustomerId", "Surname"])
leak_scores = cross_validate(rf_pipeline, X_leak, y, cv=5, scoring=['average_precision'])
print("PR-AUC with leaked data (Complain column): ", leak_scores['test_average_precision'].mean())

## Data Leakage: The "Complain" Feature

To demonstrate the impact of data leakage, the model was trained again with the `Complain` feature included.

### Results

| Configuration | PR-AUC |
|---------------|---------|
| Without Complain | ~0.67 |
| With Complain | ~0.996 |


A customer complaint is probably recorded shortly before the customer leaves the bank.
As a result, the model gains access to information that is strongly linked to the future outcome.